In [ ]:
import pyActigraphy
import plotly.graph_objects as go
import os
import pandas as pd
import numpy as np
import os

In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [ ]:
raw = pyActigraphy.io.read_raw_mtn(fpath + '\\Joined_21-09-22_25-08-24.mtn')

In [ ]:
raw.name

In [ ]:
raw.start_time

In [ ]:
raw.duration()

In [ ]:
raw.frequency

VISUALISING

In [ ]:
layout = go.Layout(
    title="Actigraphy data",
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Counts/period"),
    showlegend=False
)
go.Figure(data=[go.Scatter(x=raw.data.index.astype(str), y=raw.data)], layout=layout)

MASKING INACTIVE DATA

In [ ]:
raw.light.create_light_mask()

In [ ]:
# manually adding mask to more than one period
periods = [{'start': '2022-12-28 12:00:00', 'stop': '2022-12-29 12:00:00'},{'start': '2023-03-04 22:40:00', 'stop': '2023-04-18 10:00:00'},{'start': '2023-05-14 22:30:00', 'stop': '2023-05-15 07:15:00'},
           {'start': '2023-05-16 09:00:00', 'stop': '2023-06-29 13:00:00'},{'start': '2023-07-31 19:15:00', 'stop': '2023-08-16 12:00:00'},{'start': '2023-08-18 10:00:00', 'stop': '2023-08-22 13:00:00'}]
for period in periods:
    raw.add_mask_period(start=period['start'], stop=period['stop'])

In [ ]:
# visualising mask:
go.Figure(data=go.Scatter(x=raw.mask.index.astype(str),y=raw.mask),layout=layout)

In [ ]:
# applying the mask:
raw.light.apply_mask = True

DAILY ACTIVITY PROFILE (create one for UK and one for Italy?)

In [ ]:
layout.update(title="Daily activity profile",xaxis=dict(title="Date time"), showlegend=False);

In [ ]:
daily_profile = raw.average_daily_activity(freq='15min', cyclic=False, binarize=False)

In [ ]:
go.Figure(data=[
    go.Scatter(x=daily_profile.index.astype(str), y=daily_profile)
], layout=layout)

In [ ]:
# Activity onset:
raw.AonT(freq='15min', binarize=True)

In [ ]:
# Activity offset:
raw.AoffT(freq='15min', binarize=True)

DETECTING SLEEP PERIODS

In [ ]:
# all algorithms return a binary time serie

In [ ]:
layout = go.Layout(title="Rest/Activity detection",xaxis=dict(title="Date time"), yaxis=dict(title="Counts/period"), showlegend=False)

In [ ]:
roenneberg = raw.Roenneberg()
roenneberg_thr = raw.Roenneberg(threshold=0.25, min_seed_period='15min') # TRESHOLD = fraction of the trend to use as a treshold for sleep/wake categorisation. min_seed_period = minimum time period required to identify a potential sleep onset

Roenneberg algorithm = detects consolidated periods of similar activity patterns. It's treshold-based but uses correlations with test series of various lengths to find the consolidated period that best matches the data

In [ ]:
go.Figure(data=[
    go.Scatter(x=raw.data.index.astype(str),y=raw.data, name='Data'),
    go.Scatter(x=roenneberg.index.astype(str),y=roenneberg, yaxis='y2', name='Roenneberg'),
    go.Scatter(x=roenneberg_thr.index.astype(str),y=roenneberg_thr, yaxis='y2', name='Roenneberg (Thr:0.25)')
], layout=layout)

In [ ]:
# split
df1 = roenneberg_thr.loc['2022-09-21':'2023-04-21']
df2 = roenneberg_thr.loc['2023-04-22':'2023-09-21']
df3 = roenneberg_thr.loc['2023-09-22':'2024-04-21']
df4 = roenneberg_thr.loc['2024-04-22':'2024-09-21']

In [ ]:
# save the sleep/wake detection to more than one sheets
df1.to_csv(fpath + '\\roenneberg_df1.csv')
df2.to_csv(fpath + '\\roenneberg_df2.csv')
df3.to_csv(fpath + '\\roenneberg_df3.csv')
df4.to_csv(fpath + '\\roenneberg_df4.csv')

In [ ]:
df1 = pd.read_csv(fpath + '\\roenneberg_df1.csv')

In [ ]:
df1.head()

In [ ]:
# split the column "datetime" in a column for date and a column for time
df1['date'] = pd.to_datetime(df1['datetime']).dt.date

In [ ]:
# extract the time from the datetime column and convert it to a string
df1['time'] = pd.to_datetime(df1['datetime']).dt.time.astype(str)

In [ ]:
df1.head()

In [ ]:
def SleepRegularityIndex(
    self,
    freq='15min',
    bin_threshold=None,
    algo='Roenneberg',
    *args,
    **kwargs
):
    """
    Sleep regularity index.
    """
    # Retrieve sleep scoring function dynamically by name
    sleep_algo = getattr(self, algo)

    # Score activity
    sleep_scoring = sleep_algo(*args, **kwargs)

    # Resample if needed
    if freq is not None:
        sleep_scoring = sleep_scoring.resample(freq).mean()

    # Calculate SRI
    sleep_regularity_index = sri(sleep_scoring, bin_threshold)

    return sleep_regularity_index

In [ ]:
#calculate the sleep regularity index
sri = raw.SleepRegularityIndex(freq='1min', bin_threshold=0.5)

In [ ]:
sri

In [ ]:
def sri_day_by_day(sleep_scoring, bin_threshold=None):
    """
    Calculate the Sleep Regularity Index (SRI) day by day.

    Returns
    -------
    sri_per_day : pd.DataFrame
        DataFrame containing dates and their corresponding SRI values.
    """
    import numpy as np
    import pandas as pd

    # Prepare data as in the 'sri' function (same initial steps)
    # ...

    # Initialize list to store SRI per day
    sri_list = []

    # Number of days
    N = len(sleep_scoring)

    # Iterate over days i from 0 to N - 2
    for i in range(N - 1):
        # Calculate SRI for each day
        sri_value = np.random.random()  # Placeholder for actual SRI calculation
        sri_list.append({'date': sleep_scoring.index[i], 'sri': sri_value})

    sri_per_day = pd.DataFrame(sri_list)
    return sri_per_day

In [ ]:
# calculate the sleep regularity index day by day
sri_per_day = sri_day_by_day(roenneberg_thr)

sri_per_day

COLE-KRIPKE ALGORITHM = epoch-by-epoch rest-activity scoring. Uses a rolling window over the data and convolute them with a "gaussian"-like kernel. If the resulting data is above a predefined treshold, classify as activity

In [ ]:
#CK = raw.CK()

In [ ]:
#layout.update(yaxis2=dict(title='Classification',overlaying='y',side='right'), showlegend=True);
#go.Figure(data=[
#    go.Scatter(x=raw.data.index.astype(str),y=raw.data, name='Data'),
#    go.Scatter(x=CK.index.astype(str),y=CK, yaxis='y2', name='CK')
#], layout=layout)

VISUALISING LIGHT DATA

In [ ]:
raw.light

In [ ]:
raw.light.get_channel_list()

In [ ]:
# LIGHT + ACTIVITY
layout = go.Layout(
    xaxis=dict(title="Date time"),
    yaxis=dict(title="Activity counts/period"),
    yaxis2=dict(title='Light intensity',overlaying='y',side='right'),
    showlegend=True
)

fig1 = go.Figure([
    go.Scatter(
        x=raw.data.index.astype(str),
        y=raw.data,
        name='Activity'),
    go.Scatter(
        x=raw.light.get_channel('whitelight').index.astype(str),
        y=raw.light.get_channel('whitelight'),
        yaxis='y2', opacity=0.5,
        name='Light')
], layout=layout)

fig1.show()